In [ ]:
!pip install transformers datasets huggingface_hub tensorboard==2.15
!pip install keras~=2.15.0
!pip install wurlitzer
!sudo apt-get install git-lfs --yes

In [ ]:
!pip install transformers datasets evaluate accelerate peft

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    AutoConfig,
)

In [ ]:
model_id = "roberta-base"

In [ ]:
import os
import pandas as pd

# Load the original dataset
df = pd.read_csv(os.path.join('/kaggle/input/sarcasm-detection-dataset/', 'sarcasm-detection-dataset.csv'))

# Calculate the sizes for training, validation, and test sets
train_size = int(0.64 * len(df))
val_size = int(0.16 * len(df))
test_size = len(df) - train_size - val_size

# Split the dataset into train, validation, and test sets
train_df = df[:train_size]
val_df = df[train_size:train_size + val_size]
test_df = df[train_size + val_size:]

# Save train, validation, and test sets to separate CSV files
train_df[['text', 'label']].to_csv(os.path.join('/kaggle/working/', 'train.csv'), index=False)
val_df[['text', 'label']].to_csv(os.path.join('/kaggle/working/', 'val.csv'), index=False)
test_df[['text', 'label']].to_csv(os.path.join('/kaggle/working/', 'test.csv'), index=False)

In [ ]:
from datasets import load_dataset
train_dataset = load_dataset("csv", data_files = os.path.join('/kaggle/working/', 'train.csv'))
test_dataset = load_dataset("csv", data_files=os.path.join('/kaggle/working/', 'test.csv'))
val_dataset = load_dataset("csv", data_files=os.path.join('/kaggle/working/', 'val.csv'))

In [ ]:
train_dataset

In [ ]:
train_dataset=train_dataset['train']
test_dataset=test_dataset['train']
val_dataset=val_dataset['train']

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(model_id)

# This function tokenizes the input text using the RoBERTa tokenizer. 
# It applies padding and truncation to ensure that all sequences have the same length (256 tokens).
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True, batch_size=len(train_dataset))
val_dataset = val_dataset.map(tokenize, batched=True, batch_size=len(val_dataset))
test_dataset = test_dataset.map(tokenize, batched=True, batch_size=len(test_dataset))

In [ ]:
# Set dataset format
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
num_labels = 2
class_names = ["Not Sarcastic", "Sarcastic"]
print(f"number of labels: {num_labels}")
print(f"the labels: {class_names}")

# Create an id2label mapping
id2label = {i: label for i, label in enumerate(class_names)}

# Update the model's configuration with the id2label mapping
config = AutoConfig.from_pretrained(model_id)
config.update({"id2label": id2label})

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='steps',
    learning_rate=5e-5,
    num_train_epochs=1,
    per_device_train_batch_size=16,
)

In [ ]:
def get_trainer(model):
      return  Trainer(
          model=model,
          args=training_args,
          train_dataset=train_dataset,
          eval_dataset=test_dataset
      )

In [ ]:
from transformers import RobertaModel, RobertaTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
full_finetuning_trainer = get_trainer(
    AutoModelForSequenceClassification.from_pretrained(model_id, id2label=id2label),
)

full_finetuning_trainer.train()

In [ ]:
full_finetuning_trainer.evaluate()